# 🚀 Qwen 3.5-2B → LiteRT Conversion

Converts **Qwen3.5-2B** to LiteRT `.tflite` format. Run in **Google Colab**.

In [ ]:
# Step 1: Install dependencies
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cpu
!pip install --pre torchao
!pip install --pre ai-edge-litert-nightly[model-utils]
!pip install --pre ai-edge-quantizer-nightly
!pip install absl-py scipy numpy tabulate safetensors transformers multipledispatch jaxtyping fire rich huggingface_hub
print('\n✅ Dependencies installed!')

: 

In [ ]:
# Step 2: HF Auth + Clone repo
import os
os.environ['HF_TOKEN'] = 'your_hf_token_here'
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

!rm -rf /content/litert-torch
!git clone https://github.com/google-ai-edge/litert-torch.git /content/litert-torch
print('\n✅ Ready!')

In [ ]:
# Step 3: Fix torchao compatibility (patches ALL affected files)
import subprocess

# Find every file with the problematic import
result = subprocess.run(
    ['grep', '-rl', 'import torchao.quantization.pt2e.quantize_pt2e', '/content/litert-torch/'],
    capture_output=True, text=True
)

for fpath in result.stdout.strip().split('\n'):
    if not fpath:
        continue
    with open(fpath, 'r') as f:
        lines = f.readlines()
    new_lines = []
    for line in lines:
        if 'import torchao.quantization.pt2e.quantize_pt2e' in line:
            # Replace the hard import with a safe try/except block
            new_lines.append('try:\n')
            new_lines.append('    import torchao.quantization.pt2e.quantize_pt2e  # pylint: disable=unused-import\n')
            new_lines.append('except (ImportError, ModuleNotFoundError):\n')
            new_lines.append('    pass\n')
        else:
            new_lines.append(line)
    with open(fpath, 'w') as f:
        f.writelines(new_lines)
    print(f'  ✓ Patched: {fpath}')

print('\n✅ torchao compatibility fixed!')

In [ ]:
# Step 4: Patch model_config.py — Add GatedDeltaNet config types
config_path = '/content/litert-torch/litert_torch/generative/layers/model_config.py'
with open(config_path, 'r') as f:
    content = f.read()

content = content.replace(
    'LOCAL_SLIDING = enum.auto()',
    'LOCAL_SLIDING = enum.auto()\n  LINEAR_ATTENTION = enum.auto()'
)

content = content.replace(
    '@dataclasses.dataclass\nclass TransformerBlockConfig:',
    '@dataclasses.dataclass\nclass GatedDeltaNetConfig:\n  """Configuration for Gated DeltaNet (linear attention) layers."""\n  num_qk_heads: int = 16\n  num_v_heads: int = 16\n  qk_head_dim: int = 128\n  v_head_dim: int = 128\n  conv_kernel_dim: int = 4\n  use_output_gate: bool = True\n  use_bias: bool = False\n\n\n@dataclasses.dataclass\nclass TransformerBlockConfig:'
)

content = content.replace(
    'kv_cache_max_len: Optional[int] = None',
    'kv_cache_max_len: Optional[int] = None\n  gated_deltanet_config: Optional[GatedDeltaNetConfig] = None'
)

with open(config_path, 'w') as f:
    f.write(content)
print('✅ model_config.py patched!')

In [ ]:
# Step 5: Create gated_deltanet.py
with open('/content/litert-torch/litert_torch/generative/layers/gated_deltanet.py', 'w') as f:
    f.write('''"""Gated DeltaNet (linear attention) layer for Qwen3.5."""
from typing import Optional, Tuple, Union
from litert_torch.generative.layers import builder
from litert_torch.generative.layers import kv_cache as kv_utils
import litert_torch.generative.layers.model_config as cfg
import torch
from torch import nn
import torch.nn.functional as F


class GatedDeltaNetAttention(nn.Module):
  def __init__(self, dim, config, enable_hlfb=True):
    super().__init__()
    self.config, self.dim = config, dim
    self.num_qk_heads = config.num_qk_heads
    self.num_v_heads = config.num_v_heads
    self.qk_head_dim = config.qk_head_dim
    self.v_head_dim = config.v_head_dim
    self.conv_kernel_dim = config.conv_kernel_dim
    q_dim = self.num_qk_heads * self.qk_head_dim
    k_dim = self.num_qk_heads * self.qk_head_dim
    v_dim = self.num_v_heads * self.v_head_dim
    self.q_proj = nn.Linear(dim, q_dim, bias=config.use_bias)
    self.k_proj = nn.Linear(dim, k_dim, bias=config.use_bias)
    self.v_proj = nn.Linear(dim, v_dim, bias=config.use_bias)
    self.o_proj = nn.Linear(v_dim, dim, bias=config.use_bias)
    self.q_conv = nn.Conv1d(q_dim, q_dim, kernel_size=config.conv_kernel_dim, padding=config.conv_kernel_dim-1, groups=q_dim)
    self.k_conv = nn.Conv1d(k_dim, k_dim, kernel_size=config.conv_kernel_dim, padding=config.conv_kernel_dim-1, groups=k_dim)
    self.v_conv = nn.Conv1d(v_dim, v_dim, kernel_size=config.conv_kernel_dim, padding=config.conv_kernel_dim-1, groups=v_dim)
    self.beta = nn.Parameter(torch.zeros(self.num_qk_heads))
    self.A_log = nn.Parameter(torch.zeros(self.num_qk_heads))
    self.output_gate = nn.Linear(dim, v_dim, bias=config.use_bias) if config.use_output_gate else None

  def _causal_conv(self, x, conv):
    x = x.transpose(1, 2)
    x = conv(x)[:, :, :-(self.conv_kernel_dim - 1)]
    return x.transpose(1, 2)

  def _recurrent(self, q, k, v, state=None):
    B, T, H, D = q.shape
    decay = torch.sigmoid(self.A_log).view(1, H, 1, 1)
    beta = torch.sigmoid(self.beta).view(1, H, 1, 1)
    k = F.normalize(k, p=2, dim=-1)
    if state is None:
      state = torch.zeros(B, H, self.v_head_dim, D, dtype=q.dtype, device=q.device)
    outputs = []
    for t in range(T):
      q_t, k_t, v_t = q[:, t], k[:, t], v[:, t]
      state = decay * state + beta * (v_t.unsqueeze(-1) * k_t.unsqueeze(-2))
      outputs.append(torch.einsum("bhvk,bhk->bhv", state, q_t))
    return torch.stack(outputs, dim=1).reshape(B, T, -1), state

  def forward(self, x, rope=None, mask=None, input_pos=None, kv_cache=None, recurrent_state=None):
    B, T, _ = x.size()
    q = F.silu(self._causal_conv(self.q_proj(x), self.q_conv))
    k = F.silu(self._causal_conv(self.k_proj(x), self.k_conv))
    v = self._causal_conv(self.v_proj(x), self.v_conv)
    q = q.view(B, T, self.num_qk_heads, self.qk_head_dim)
    k = k.view(B, T, self.num_qk_heads, self.qk_head_dim)
    v = v.view(B, T, self.num_v_heads, self.v_head_dim)
    output, new_state = self._recurrent(q, k, v, recurrent_state)
    if self.output_gate is not None:
      output = output * torch.sigmoid(self.output_gate(x))
    y = self.o_proj(output)
    return y if recurrent_state is None else (y, new_state)


class HybridTransformerBlock(nn.Module):
  def __init__(self, config, model_config):
    super().__init__()
    self.pre_atten_norm = builder.build_norm(model_config.embedding_dim, config.pre_attention_norm_config)
    self.post_atten_norm = builder.build_norm(model_config.embedding_dim, config.post_attention_norm_config)
    self.ff = builder.build_ff(model_config.embedding_dim, config.ff_config)
    self.config = config
    self.is_linear_attention = config.gated_deltanet_config is not None
    if self.is_linear_attention:
      self.atten_func = GatedDeltaNetAttention(model_config.embedding_dim, config.gated_deltanet_config)
    else:
      from litert_torch.generative.layers import attention
      self.atten_func = attention.CausalSelfAttention(model_config.embedding_dim, config.attn_config, model_config.enable_hlfb)

  def forward(self, x, rope=None, mask=None, input_pos=None, kv_cache=None, recurrent_state=None):
    x_norm = self.pre_atten_norm(x)
    if self.is_linear_attention:
      res = self.atten_func(x_norm, recurrent_state=recurrent_state)
      if recurrent_state is not None:
        attn_out, new_state = res
      else:
        attn_out, new_state = res, None
    else:
      res = self.atten_func(x_norm, rope, mask, input_pos, kv_cache)
      if kv_cache is not None:
        attn_out, kv = res
      else:
        attn_out, kv = res, None
      new_state = None
    x = x + attn_out
    output = x + self.ff(self.post_atten_norm(x))
    if self.is_linear_attention and new_state is not None:
      return output, new_state
    if not self.is_linear_attention and kv_cache is not None:
      return output, kv
    return output
''')
print('✅ gated_deltanet.py created!')

In [ ]:
# Step 6: Create qwen3_5.py
with open('/content/litert-torch/litert_torch/generative/examples/qwen/qwen3_5.py', 'w') as f:
    f.write('''"""Qwen 3.5 model definition."""
from typing import Callable, Dict, Optional
import litert_torch.generative.layers.model_config as cfg
from litert_torch.generative.layers import builder, gated_deltanet
from litert_torch.generative.layers import kv_cache as kv_utils
from litert_torch.generative.utilities import loader as loading_utils
import litert_torch.generative.layers.attention_utils as attn_utils
from litert_torch.generative.utilities import export_config as export_cfg
import torch
from torch import nn

LAYER_TYPES = (["linear_attention"] * 3 + ["full_attention"]) * 6

TENSOR_NAMES = loading_utils.ModelLoader.TensorNames(
    ff_up_proj="model.layers.{}.mlp.up_proj",
    ff_down_proj="model.layers.{}.mlp.down_proj",
    ff_gate_proj="model.layers.{}.mlp.gate_proj",
    attn_query_proj="model.layers.{}.self_attn.q_proj",
    attn_key_proj="model.layers.{}.self_attn.k_proj",
    attn_value_proj="model.layers.{}.self_attn.v_proj",
    attn_output_proj="model.layers.{}.self_attn.o_proj",
    pre_attn_norm="model.layers.{}.input_layernorm",
    post_attn_norm="model.layers.{}.post_attention_layernorm",
    embedding="model.embed_tokens",
    final_norm="model.norm",
)

def get_2b_model_config():
  norm = cfg.NormalizationConfig(type=cfg.NormalizationType.RMS_NORM, epsilon=1e-06)
  dn = cfg.GatedDeltaNetConfig(num_qk_heads=16, num_v_heads=16, qk_head_dim=128, v_head_dim=128, conv_kernel_dim=4)
  full_attn = cfg.AttentionConfig(num_heads=8, head_dim=256, num_query_groups=2, rotary_base=10000000, rotary_percentage=0.25, qkv_use_bias=False, qkv_transpose_before_split=True, attn_type=cfg.AttentionType.GLOBAL)
  lin_attn = cfg.AttentionConfig(num_heads=16, head_dim=128, num_query_groups=16, rotary_percentage=0.0, attn_type=cfg.AttentionType.LINEAR_ATTENTION)
  ff = cfg.FeedForwardConfig(type=cfg.FeedForwardType.GATED, activation=cfg.ActivationConfig(cfg.ActivationType.SILU), intermediate_size=6144)
  blocks = [cfg.TransformerBlockConfig(attn_config=lin_attn if t=="linear_attention" else full_attn, ff_config=ff, pre_attention_norm_config=norm, post_attention_norm_config=norm, gated_deltanet_config=dn if t=="linear_attention" else None) for t in LAYER_TYPES]
  return cfg.ModelConfig(vocab_size=248320, num_layers=24, max_seq_len=2048, embedding_dim=2048, block_configs=blocks, final_norm_config=norm, lm_head_share_weight_with_embedding=True)

class Qwen3_5(nn.Module):
  def __init__(self, config, mask_cache_size=0):
    super().__init__()
    self.tok_embedding = nn.Embedding(config.vocab_size, config.embedding_dim, padding_idx=0)
    self.lm_head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)
    if config.lm_head_share_weight_with_embedding:
      self.lm_head.weight.data = self.tok_embedding.weight.data
    self.transformer_blocks = nn.ModuleList([gated_deltanet.HybridTransformerBlock(config.block_config(i), config) for i in range(config.num_layers)])
    self.final_norm = builder.build_norm(config.embedding_dim, config.final_norm_config)
    self.config = config
    self.is_linear = [config.block_config(i).gated_deltanet_config is not None for i in range(config.num_layers)]
    self.mask_cache = attn_utils.build_causal_mask_cache(mask_cache_size) if mask_cache_size > 0 else None

  @torch.inference_mode
  def forward(self, tokens, input_pos, kv_cache, mask=None, export_config=None):
    x = self.tok_embedding(tokens)
    full_idx = next(i for i, lin in enumerate(self.is_linear) if not lin)
    ac = self.config.block_config(full_idx).attn_config
    rope = self.config.build_rope(input_pos, int(ac.rotary_percentage * ac.head_dim), ac.rotary_base)
    if mask is None and self.mask_cache is not None and kv_cache is not None:
      mask = self.mask_cache.index_select(2, input_pos)[:, :, :, :kv_cache.get_max_seq_len()]
    kv_idx, updated_kv = 0, []
    for i, block in enumerate(self.transformer_blocks):
      if self.is_linear[i]:
        x = block(x)
      else:
        kv_entry = kv_cache.caches[kv_idx] if kv_cache else None
        x, kv_entry = block(x, rope, mask, input_pos, kv_entry)
        if kv_entry: updated_kv.append(kv_entry)
        kv_idx += 1
    new_kv = kv_utils.KVCache(tuple(updated_kv))
    if export_config is not None:
      if torch.numel(input_pos) > 1 and not export_config.output_logits_on_prefill:
        return {"kv_cache": new_kv}
    return {"logits": self.lm_head(self.final_norm(x)), "kv_cache": new_kv}

def build_2b_model(checkpoint_path, custom_loader=None, mask_cache_size=0):
  config = get_2b_model_config()
  model = Qwen3_5(config, mask_cache_size)
  if checkpoint_path:
    loader = loading_utils.ModelLoader(checkpoint_path, TENSOR_NAMES, custom_loader)
    loader.load(model, strict=False)
  return model.eval()
''')
print('✅ qwen3_5.py created!')

In [ ]:
# Step 7: Create conversion script
with open('/content/litert-torch/litert_torch/generative/examples/qwen/convert_v3_5_to_tflite.py', 'w') as f:
    f.write('''"""Convert Qwen 3.5 to tflite."""
from absl import app
from litert_torch.generative.examples.qwen import qwen3_5
from litert_torch.generative.utilities import converter
flags = converter.define_conversion_flags("qwen3_5")
_MODEL_SIZE = flags.DEFINE_enum("model_size", "2b", ["2b"], "Model size.")
_BUILDER = {"2b": qwen3_5.build_2b_model}
def main(_):
  converter.build_and_convert_to_tflite_from_flags(_BUILDER[_MODEL_SIZE.value])
if __name__ == "__main__":
  app.run(main)
''')
print('✅ convert_v3_5_to_tflite.py created!')

In [ ]:
# Step 8: Verify all patches
import litert_torch.generative.layers.model_config as cfg
print(f'✓ LINEAR_ATTENTION = {cfg.AttentionType.LINEAR_ATTENTION}')
print(f'✓ GatedDeltaNetConfig = {cfg.GatedDeltaNetConfig}')
from litert_torch.generative.layers.gated_deltanet import GatedDeltaNetAttention, HybridTransformerBlock
print('✓ GatedDeltaNetAttention + HybridTransformerBlock imported')
from litert_torch.generative.examples.qwen.qwen3_5 import get_2b_model_config
config = get_2b_model_config()
lin = sum(1 for i in range(24) if config.block_config(i).gated_deltanet_config is not None)
print(f'✓ {lin} linear + {24-lin} full attention layers')
print('\n✅ All patches verified!')

In [ ]:
# Step 9: Download model and convert
import os
from huggingface_hub import snapshot_download
model_path = snapshot_download('Qwen/Qwen3.5-2B', token=os.environ['HF_TOKEN'], ignore_patterns=['*.bin','*.msgpack','*.onnx'])
print(f'\n✅ Downloaded to: {model_path}')

!mkdir -p /content/output
!python -m litert_torch.generative.examples.qwen.convert_v3_5_to_tflite \
    --checkpoint_path={model_path} \
    --output_path=/content/output/ \
    --quantize=dynamic_int8 \
    --kv_cache_max_len=2048

In [ ]:
# Step 10: Check output
import glob, os
for f in glob.glob('/content/output/*'):
    print(f'  📄 {os.path.basename(f)} — {os.path.getsize(f)/(1024*1024):.1f} MB')
print('\n🎉 Done!' if glob.glob('/content/output/*') else '\n⚠️ No output found')